In [4]:
import pandas as pd
import os
import pyterrier as pt


In [5]:
MEASURES=["nDCG@10"]
FRIENDLY_MODEL = {
    'star' : 'STAR',
    'tct_colbert' : 'TCT',
    'tas_b' : 'TAS-B',
    'dragon' : 'DRAGON',
    'repllama' : 'RepLLama',
    'lion' : 'LION', 
    "e5" : "E5"
}

index_name = {
    "tct_colbert" : "tct-hnp",
    "tas_b" : "tasb",
    "repllama" : "repllama7b"
}

QRELS={
    'test-2019' : pt.get_dataset("msmarco_passage").get_qrels('test-2019'),
    'test-2020' : pt.get_dataset("msmarco_passage").get_qrels('test-2020'),
    'dev' : pt.get_dataset("irds:msmarco-passage/dev/small").get_qrels(),
}

def delta(old, new):
    return ((new - old) / old) * 100

def total_f4_size(directory):
    return sum(
        os.path.getsize(os.path.join(directory, f))
        for f in os.listdir(directory)
        if f.endswith(".f4") and os.path.isfile(os.path.join(directory, f))
    )

def bytes_to_gb(num_bytes):
    return num_bytes / (1024 ** 3)



def get_training_log(filename, need_dev=True):
    import re, ast
    pattern = re.compile(r"since step (\d+)")
    steps = None
    total_time = None
    best_val_metrics = None
    with open(filename) as f:
        for line in f:
            if "[val] at step" in line:
                match = re.search(r"(\{.*\})", line)
                last_val_metrics = ast.literal_eval(match.group(1))
            elif "New best model at step" in line:
                best_val_metrics = last_val_metrics
            elif "Early stopping at" in line:
                #print(line)
                match = pattern.search(line)
                steps = int(match.group(1))
            elif match := re.search(r"Total training time ([\d.]+) seconds", line):
                total_time = float(match.group(1))
    if steps is None and need_dev:
        raise ValueError("did not find stopping in %s" % filename)
    return steps, total_time, best_val_metrics


import ir_measures
def calc_significance(filenameA : str, filenameB : str, qrels, measure):
    if isinstance(measure, str):
        measure = ir_measures.parse_measure(measure)
    df = pt.Experiment(
        [pt.io.read_results(filenameA), pt.io.read_results(filenameB)], 
        qrels[['qid']],
        qrels,
        [measure],
        baseline=0)
    return df.iloc[1][str(measure) + " p-value"]

In [6]:
MODEL="star"

print(FRIENDLY_MODEL[MODEL] + " & Reported \cite{adore} & 0.340 & 0.642 & - \\\\ ")
print(FRIENDLY_MODEL[MODEL] + " PQ & Reported \cite{jpq} & 0.305 & 0.594 & - \\\\")

print(FRIENDLY_MODEL[MODEL] + " JPQ & Reported \cite{jpq} & 0.341 & 0.677 & 0.671 $\\square$ \\\\")


JPQRUN=f"basesetting/msmarco-passage-train__{MODEL}__faiss2opq__M96_nbits8__ps159744__neg200____lr__"
JPQLOG=f"basesetting/{MODEL}_m96_nbits8_no-ibn_jpqneg200_lr.log"

PQRUN=f"pq_only/msmarco-passage-train__{MODEL}__faiss2opq__M96_nbits8__ps159744____ibn____"
PQLOG=f"pq_only/{MODEL}_m96_nbits8.log"



row_values=[]
print(f"{FRIENDLY_MODEL[MODEL]} & Reproduction & ", end=" ")
for set,measure in [("dev", "RR@10"), ("trec2019", "nDCG@10"), ("trec2020", "nDCG@10")]:
    metrics = pd.read_csv(os.path.join("../indices/baselines", set+"_"+ MODEL, "metrics.csv"))
    row_values.append("%0.3f" % metrics.iloc[0][measure])
print(" & ".join( row_values )+ " \\\\")


def _render_row(model, name, rundir, log, baselinedir=None):
    print(f"{FRIENDLY_MODEL[model]} {name} & Reproduction & ", end=" ")
    row_values = []


    sets = [ ("test-2019", "nDCG@10"), ("test-2020", "nDCG@10")]
    if os.path.exists(os.path.join(rundir, "runs", "dev", "metrics.csv")):
        sets = [("dev", "RR@10")] + sets
    else:
        steps, time, val_metrics = get_training_log(log)
        row_values.append("%0.3f" % val_metrics["RR@10"])
    
    for set, measure in sets:
        metrics = pd.read_csv(os.path.join(rundir, "runs", set, "metrics.csv"))
        value = metrics.iloc[1][measure]

        sigBASE = ""
        pvalue = calc_significance(os.path.join(rundir, "runs", set, "baseline.res.gz"), os.path.join(rundir, "runs", set, "JPQ pq.res.gz"), QRELS[set], measure)
        if pvalue < 0.05:
            sigBASE = "$\dagger$"

        sigPQ = ""
        if baselinedir is not None:
            pvalue = calc_significance(
                os.path.join(baselinedir, "runs", set, "JPQ pq.res.gz"),  
                os.path.join(rundir, "runs", set, "JPQ pq.res.gz"), 
                QRELS[set], 
                measure)
            if pvalue < 0.05:
                sigPQ = "*"
        
        row_values.append("%0.3f%s%s" % (value, sigBASE, sigPQ))
        
    print(" & ".join( row_values )+ " \\\\")

_render_row(MODEL, "PQ", PQRUN, PQLOG)
_render_row(MODEL, "JPQ", JPQRUN, JPQLOG, baselinedir=PQRUN)


STAR & Reported \cite{adore} & 0.340 & 0.642 & - \\ 
STAR PQ & Reported \cite{jpq} & 0.305 & 0.594 & - \\
STAR JPQ & Reported \cite{jpq} & 0.341 & 0.677 & 0.671 $\square$ \\
STAR & Reproduction &  0.300 & 0.625 & 0.596 \\
STAR PQ & Reproduction &  0.243$\dagger$ & 0.525$\dagger$ & 0.535$\dagger$ \\
STAR JPQ & Reproduction &  0.271 & 0.585$\dagger$* & 0.559$\dagger$ \\


In [7]:

    
def print_model_table(models, Ms=[96], nbits=[8], pq=True, show_nbits=None, baseline=True, index_size=True, add_pad=True):

    for model in models:

        baseindex = f"../indices/{index_name.get(model, model)}"
        sizeBASE = total_f4_size(baseindex)
        
        if baseline:
            print(FRIENDLY_MODEL[model], end=" & Flat & ")
            if show_nbits:
                print(" & &", end=" ")
            if index_size:
                print(" %0.2f &  & " % bytes_to_gb(sizeBASE), end=" ")
            #else:
            #    print(" & ", end=" ")

        sets = [("dev", "dev", "RR@10"), ("trec2019", "test-2019", "nDCG@10"), ("trec2020", "test-2020", "nDCG@10")]
        row_values=[]
       

        baselines={}
        for set, setname, measure in sets:
            metrics = pd.read_csv(os.path.join("../indices/baselines",  f"{set}_{index_name.get(model, model)}", "metrics.csv"))
            value = metrics.iloc[0][measure] # 0 for baselines
            baselines[setname] = value
            values = ["%0.3f & " % v for v in [value]] 
            row_values.extend(values)

        if baseline:
            print("& ".join(row_values), end ="\\\\ ")
            print()
        
        settings = ["jpq"]
        if pq:
            settings = ["pq", "jpq"]
        for M in Ms:
            for nb in nbits: 
                baseline_files = {}
                for setting in settings:
                    baseline_files[setting] = {}
                    if setting == "jpq":                    
                        # NOTE frozen query encoder repllama and lion
                        if model == "repllama":
                            jpqrun=f"repllama/msmarco-passage-train__{model}__faiss2opq__M{M}_nbits{nb}__ps159744__neg200__ibn__lr__frozen"
                            log=f"repllama/m{M}_nbits{nb}_ibn_pqss159k-faiss2opq-lr-200jpqegs.log"
                        elif model == "lion":
                            jpqrun=f"lion/msmarco-passage-train__{model}__faiss2opq__M{M}_nbits{nb}__ps159744__neg200__ibn__lr__frozen"
                            log=f"lion/m{M}_nbits{nb}_ibn_pqss159k-faiss2opq-lr-200jpqegs.log"
                        else:
                            jpqrun=f"basesetting/msmarco-passage-train__{model}__faiss2opq__M{M}_nbits{nb}__ps159744__neg200__ibn__lr__"
                            log=f"basesetting/{model}_m{M}_nbits{nb}_ibn_jpqneg200_lr.log"
                    else:
                        jpqrun = f"pq_only/msmarco-passage-train__{model}__faiss2opq__M{M}_nbits{nb}__ps159744____ibn____"
                        log = f"pq_only/{model}_m{M}_nbits{nb}.log"

                    sets = [("dev", "RR@10"), ("test-2019", "nDCG@10"), ("test-2020", "nDCG@10")]
                    row_values=[]

                    sets = [ ("test-2019", "nDCG@10"), ("test-2020", "nDCG@10")]
                    if os.path.exists(os.path.join(jpqrun, "runs", "dev", "metrics.csv")):
                        sets = [("dev", "RR@10")] + sets
                    else:
                        #print(os.path.join(jpqrun, "runs", "dev", "metrics.csv"))
                        try:
                            steps, time, val_metrics = get_training_log(log)
                            value = val_metrics["RR@10"]
                            delta_val = delta(baselines["dev"], value)
                            row_values.append("%0.3f & \\small{%0.1f\\%%} " % (value, delta_val) )
                        except:
                            row_values.append(" ?? & ")
                            
                        
            
                    for set, measure in sets:
                        runfile = os.path.join(jpqrun, "runs", set, "JPQ pq.res.gz")
                        
                        baseline_files[setting][set] = runfile
                        metrics = pd.read_csv(os.path.join(jpqrun, "runs", set, "metrics.csv"))
                        value = metrics.iloc[1][measure]
                        delta_val = delta(baselines[set], value)
                        sigBASE = ""
                        pvalue = calc_significance(os.path.join(jpqrun, "runs", set, "baseline.res.gz"), runfile, QRELS[set], measure)
                        if pvalue < 0.05:
                            sigBASE = "$\dagger$"
                        sigPQ = ""
                        if setting == "jpq":
                            pvalue = calc_significance(baseline_files['pq'][set],  baseline_files['jpq'][set], QRELS[set], measure)
                            if pvalue < 0.05:
                                sigPQ = "*"
                        row_values.append("%0.3f%s%s  & \\small{%0.1f\\%%} " % (value, sigBASE,sigPQ, delta_val))
                        
                    
                    rowname = f"{FRIENDLY_MODEL[model]} & {setting.upper()}" 
                    if show_nbits or len(Ms) > 1:
                        rowname += f" & {M}"
                    if show_nbits or len(nbits) > 1:
                        rowname += f"& {nb}"
                    print(rowname, end=" ")
                
                    sizeJPQ = total_f4_size(jpqrun)
                    if index_size:
                        print(" & %0.2f & \\small{%d$\\times$}" % (bytes_to_gb(sizeJPQ), sizeBASE/sizeJPQ), end=" & ")
                    else:
                        print(" & ", end="")
                    print(" & ".join(row_values), end=' \\\\ ')
                    print()
        if add_pad:
            print("\\addlinespace")          
                    

In [ ]:
print_model_table(["star", "tct_colbert", "tas_b", "e5", "dragon"], index_size=False)#, 

STAR & Flat & 0.300 & & 0.625 & & 0.596 & \\ 
STAR & PQ  & 0.243$\dagger$  & \small{-19.0\%}  & 0.525$\dagger$  & \small{-16.0\%}  & 0.535$\dagger$  & \small{-10.2\%}  \\ 
STAR & JPQ  & 0.241$\dagger$  & \small{-19.7\%}  & 0.574$\dagger$  & \small{-8.2\%}  & 0.559  & \small{-6.3\%}  \\ 
\addlinespace
TCT & Flat & 0.359 & & 0.718 & & 0.684 & \\ 


In [ ]:
print_model_table(["lion"], Ms=[128,256], nbits=[8], show_nbits=True, baseline=True, add_pad=False)
print_model_table(["lion"], Ms=[512], nbits=[8,9,10], show_nbits=True, baseline=False, add_pad=True)
print_model_table(["repllama"], Ms=[128,256], nbits=[8], show_nbits=True, baseline=True, add_pad=False)
print_model_table(["repllama"], Ms=[512], nbits=[8,9,10], show_nbits=True, baseline=False, add_pad=False)

## ablation table

In [ ]:
ablation_dir = "./ablation_amd_3090"



def analyse(model, ibn=True, lr=True, frozen=True, jpqnegs=True, pq_only=False, time=False):
    if pq_only:
        logrun = f"pq_only/{model}_m96_nbits8.log"
        jpqrun = f"pq_only/msmarco-passage-train__{model}__faiss2opq__M96_nbits8__ps159744____ibn____"
    else:
        lrS1 = "lr" if lr else "nolr"
        lrS2 = "lr" if lr else ""
        frozenS1 = "frozen" if frozen else "unfrozen"
        frozenS2 = "frozen" if frozen else ""
        ibnS1 = "ibn" if ibn else "no-ibn"
        ibnS2 = "ibn" if ibn else ""
        jpgnegsS1 = "jpqneg200" if jpqnegs else "nojpqnegs"  
        jpgnegsS2 = "neg200" if jpqnegs else ""    
        logrun = os.path.join(ablation_dir, 
                              f"{model}_m96_nbits8_{ibnS1}_{jpgnegsS1}_{lrS1}_{frozenS1}.log")
        jpqrun = os.path.join(ablation_dir, 
                              f"msmarco-passage-train__{model}__faiss2opq__M96_nbits8__ps159744__{jpgnegsS2}__{ibnS2}__{lrS2}__{frozenS2}")
        pqrun = f"pq_only/msmarco-passage-train__{model}__faiss2opq__M96_nbits8__ps159744____ibn____"
    logexists = os.path.exists(logrun)
    if not os.path.exists(jpqrun):
        raise ValueError("jpqrun %s does not exist; logrun %s exists %r" % (jpqrun, logrun, logexists))
    if not logexists:
        raise ValueError("jpqrun %s exist, but logrun %s does not" % (jpqrun, logrun))

    sets = [ ("test-2019", "nDCG@10"), ("test-2020", "nDCG@10")]
    row_values=[]
    if os.path.exists(os.path.join(jpqrun, "runs", "dev", "metrics.csv")):
        sets = [("dev", "RR@10")] + sets
        steps, total_time, _ = get_training_log(logrun, need_dev=False)
    else:
        steps, total_time, val_metrics = get_training_log(logrun, need_dev=True)
        row_values.append("%0.3f" % val_metrics["RR@10"])
    
    for set, measure in sets:
        metrics = pd.read_csv(os.path.join(jpqrun, "runs", set, "metrics.csv"))
        value = metrics.iloc[1][measure]
        

        sigBASE = ""
        pvalue = calc_significance(os.path.join(jpqrun, "runs", set, "baseline.res.gz"), os.path.join(jpqrun, "runs", set, "JPQ pq.res.gz"), QRELS[set], measure)
        if pvalue < 0.05:
            sigBASE = "$\dagger$"

        sigPQ = ""
        if not pq_only:
            pvalue = calc_significance(
                os.path.join(pqrun, "runs", set, "JPQ pq.res.gz"),  
                os.path.join(jpqrun, "runs", set, "JPQ pq.res.gz"), 
                QRELS[set], 
                measure)
            if pvalue < 0.05:
                sigPQ = "*"
        
        row_values.append("%0.3f%s%s" % (value, sigBASE, sigPQ))

    if pq_only:
        steps = " "
        
    # metricsA = pd.read_csv(os.path.join(jpqrun, "runs",  "test-2019", "metrics.csv"))
    # metricsB = pd.read_csv(os.path.join(jpqrun, "runs",  "test-2020", "metrics.csv"))

    #steps, total_time, best_val = get_training_log(logrun)

    total_time = "%0.1f" % (total_time / 3600.)
    #ndcgs = [best_val["RR@10"], metricsA.iloc[1]["nDCG@10"],  metricsB.iloc[1]["nDCG@10"] ]
    #ndcgs = ["%0.4f" % n for n in ndcgs]
    if time:
        return steps, total_time, *row_values
    else:
        return steps, *row_values

def render(model, time=False, **kwargs):
    metrics = analyse(model, time=time **kwargs)
    metrics = [str(m) for m in metrics]
    if kwargs.get("pq_only", False):
        if time:
            print("PQ only & & & & ", end="")
        else:
            print("PQ only & & & ", end="")
    else:
        if kwargs["frozen"]:
            print("Frozen", end = " & ")
        else:
            print("Trained", end = " & ")
        for k in ["ibn", "lr",  'jpqnegs']:
            if kwargs[k]:
                print("\\cmark", end = " & ")
            else:
                print("\\xmark", end = " & ")
    print( " & ".join(metrics), end="\\\\ \n")

In [ ]:
SETTINGS = [
 {'frozen': True, 'ibn': False, 'lr': False, 'jpqnegs': False},
 {'frozen': False, 'ibn': False, 'lr': False, 'jpqnegs': False},
 {'frozen': False, 'ibn': False, 'lr': False, 'jpqnegs': True}, 
 #{'frozen': False, 'ibn': False, 'lr': True, 'jpqnegs': False}, -- cant run lr without jpqnegs 
 {'frozen': False, 'ibn': False, 'lr': True, 'jpqnegs': True},
 {'frozen': False, 'ibn': True, 'lr': False, 'jpqnegs': False},
 {'frozen': False, 'ibn': True, 'lr': False, 'jpqnegs': True}, 
 #{'frozen': False, 'ibn': True, 'lr': True, 'jpqnegs': False}, -- cant run lr without jpqnegs 
 {'frozen': False, 'ibn': True, 'lr': True, 'jpqnegs': True},
 {'frozen': True, 'ibn': True, 'lr': True, 'jpqnegs': True}
]

render("tct_colbert", pq_only=True)
for i, setting in enumerate(SETTINGS):
    try:
        render("tct_colbert", **setting)
    except Exception as e:
        print("skipping %d %s" % (i, str(setting)), e)